# 03 — Évaluation finale sur le holdout temporel

Ce notebook constitue l'**évaluation finale** du système figé sur les steps **378 à 743**.

> **Règle de gouvernance :** aucun résultat de ce notebook ne doit servir à modifier les features, les modèles, la calibration, l'architecture H2 ou la politique économique.

Le Test est décrit comme un **holdout final gelé à partir du reset méthodologique**. Il n'est pas présenté comme historiquement jamais vu, car une ancienne version du projet avait déjà exploré PaySim.

### Artefacts évalués

**H2 v1**  
`6d5d5dea6d73d6f200ab2206f1aaeb5376e044b689267321acdd984d60e3ef03`

**Economic probability v1**  
`77abc391b911c2c07e5b6c9410a0cd810ce898d887b4a8f3d99c929d4b6c16ed`


## 1. Chargement du Test et contrat d'intégrité


In [1]:
# Charger le Test directement depuis PostgreSQL
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)

from src.config import (
    DATABASE_URL,
    MODELS_DIR,
)

from src.features import (
    build_static_features,
    prepare_xgb_matrix,
)

from src.scoring import (
    exact_drain_mask,
    hybrid_priority_score,
    calibrated_fraud_probability,
)

from src.policy import (
    expected_net_value,
)

from src.metrics import (
    precision_recall_at_k,
)

In [2]:
# Connexion

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
)

In [3]:
# Requête
TEST_QUERY = text("""
    SELECT
        step,
        type,
        amount::double precision AS amount,
        oldbalance_org::double precision AS oldbalance_org,
        oldbalance_dest::double precision AS oldbalance_dest,
        is_fraud::smallint AS is_fraud
    FROM transactions_raw
    WHERE step > 377
    ORDER BY step
""")

In [4]:
with engine.connect() as connection:
    test = pd.read_sql_query(
        TEST_QUERY,
        connection,
    )

engine.dispose()

In [5]:
# Contrat d'intégrité avant scoring
assert len(test) == 955_744

assert test["step"].min() == 378
assert test["step"].max() == 743

assert test["is_fraud"].sum() == 4_010

assert test.isna().sum().sum() == 0

print("TEST final chargé : intégrité OK")

TEST final chargé : intégrité OK


### Conclusion — holdout chargé sans mélange

Le Test est chargé directement depuis PostgreSQL, puis contrôlé avant scoring. Les labels sont utilisés uniquement pour l'évaluation finale, pas pour construire de nouvelles features ou décisions.


## 2. Évaluation finale du moteur H2


In [6]:
# Charger H2 depuis l'artefact gelé
h2_artifact = joblib.load(
    MODELS_DIR
    / "h2_residual_xgb_v1.joblib"
)

h2_model = h2_artifact[
    "residual_model"
]

h2_columns = h2_artifact[
    "xgb_columns"
]

In [7]:
# Features
X_test = build_static_features(
    test
)

X_test_h2 = prepare_xgb_matrix(
    X_test
)

X_test_h2 = X_test_h2.drop(
    columns=["log_ecart_vidage"],
    errors="ignore",
)

X_test_h2 = X_test_h2.reindex(
    columns=h2_columns,
    fill_value=0.0,
)

In [8]:
# Score résiduel
h2_residual_score = (
    h2_model.predict_proba(
        X_test_h2
    )[:, 1]
)

In [10]:
# Score hybride
h2_test_score = (
    hybrid_priority_score(
        test,
        h2_residual_score,
    )
)

In [11]:
# Métrique principale H2
h2_test_ap = average_precision_score(
    test["is_fraud"],
    h2_test_score,
)

test_prevalence = (
    test["is_fraud"].mean()
)

print(
    f"Test prevalence : "
    f"{test_prevalence:.6%}"
)

print(
    f"H2 Test AP      : "
    f"{h2_test_ap:.6f}"
)

print(
    f"H2 Test AP lift : "
    f"{h2_test_ap / test_prevalence:.2f}x"
)

Test prevalence : 0.419568%
H2 Test AP      : 0.997873
H2 Test AP lift : 237.83x


In [14]:
# Precision@K / Recall@K
h2_test_at_k = pd.DataFrame([
    precision_recall_at_k(
        test["is_fraud"],
        h2_test_score,
        k,
    )
    for k in [
        100,
        500,
        1_000,
        5_000,
        10_000,
    ]
])

h2_test_at_k

,k,tp,precision,recall
0,100,100,1.0000,0.024938
1,500,500,1.0000,0.124688
2,1000,1000,1.0000,0.249377
3,5000,3997,0.7994,0.996758
4,10000,4001,0.4001,0.997756


### Conclusion — H2 conserve un ranking très fort hors période

Le Test contient **955 744 transactions** et **4 010 fraudes**, soit une prévalence d'environ **0,4196 %**, nettement supérieure à Validation.

H2 atteint :
- **Average Precision = 0,997873**
- **Precision@100 = 100 %**
- **Precision@1 000 = 100 %**
- **Recall@5 000 = 99,68 %**

La performance de ranking reste donc très élevée malgré le changement temporel de prévalence.


## 3. Contribution de la règle de vidage exact


In [15]:
# Évaluer séparément B2a
exact_test = exact_drain_mask(
    test
)

exact_alerts = int(
    exact_test.sum()
)

exact_tp = int(
    test.loc[
        exact_test,
        "is_fraud"
    ].sum()
)

exact_precision = (
    exact_tp / exact_alerts
    if exact_alerts else 0
)

exact_recall = (
    exact_tp
    / test["is_fraud"].sum()
)

print(
    f"B2a alertes   : {exact_alerts:,}"
)

print(
    f"B2a TP        : {exact_tp:,}"
)

print(
    f"B2a Precision : {exact_precision:.2%}"
)

print(
    f"B2a Recall    : {exact_recall:.2%}"
)

B2a alertes   : 3,896
B2a TP        : 3,896
B2a Precision : 100.00%
B2a Recall    : 97.16%


### Conclusion — la règle reste forte, mais le résiduel est nécessaire

La règle exacte seule détecte **3 896 fraudes sur 4 010** :
- Precision = **100 %**
- Recall ≈ **97,16 %**

Elle manque donc **114 fraudes**. Le gain de H2 ne consiste pas à remplacer la règle, mais à **prioriser les exceptions** qu'elle ne couvre pas.


## 4. Évaluation du moteur probabiliste et de la calibration hors période


In [19]:
# Évaluer le modèle économique gelé
economic_artifact = joblib.load(
    MODELS_DIR
    / "economic_probability_v1.joblib"
)

economic_model = (
    economic_artifact[
        "base_model"
    ]
)

economic_calibrator = (
    economic_artifact[
        "calibrator"
    ]
)

economic_columns = (
    economic_artifact[
        "xgb_columns"
    ]
)

# Matrice
X_test_economic = prepare_xgb_matrix(
    X_test
)

X_test_economic = X_test_economic.reindex(
    columns=economic_columns,
    fill_value=0.0,
)

# Probabilité brute
test_probability_raw = (
    economic_model.predict_proba(
        X_test_economic
    )[:, 1]
)

# Probabilité calibrée 
test_probability = (
    calibrated_fraud_probability(
        test_probability_raw,
        economic_calibrator,
    )
)

In [20]:
# Calibration finale hors période
test_brier = brier_score_loss(
    test["is_fraud"],
    test_probability,
)

test_logloss = log_loss(
    test["is_fraud"],
    test_probability,
)

test_prob_ap = (
    average_precision_score(
        test["is_fraud"],
        test_probability,
    )
)

print(
    f"Probability mean : "
    f"{test_probability.mean():.8f}"
)

print(
    f"Observed rate    : "
    f"{test_prevalence:.8f}"
)

print(
    f"Brier Test      : "
    f"{test_brier:.8f}"
)

print(
    f"Log Loss Test   : "
    f"{test_logloss:.8f}"
)

print(
    f"AP probability  : "
    f"{test_prob_ap:.6f}"
)

Probability mean : 0.00338761
Observed rate    : 0.00419568
Brier Test      : 0.00076047
Log Loss Test   : 0.00235014
AP probability  : 0.975560


### Conclusion — excellente discrimination, calibration à surveiller

Sur Test :
- Average Precision probabiliste = **0,975560**
- Brier Score = **0,00076047**
- Log Loss = **0,00235014**
- probabilité moyenne prédite = **0,00338761**
- fréquence observée = **0,00419568**

La probabilité moyenne **sous-estime le risque observé d'environ 19 % en relatif**. Le moteur reste discriminant, mais ce décalage confirme qu'une calibration doit être suivie dans le temps lorsque la prévalence évolue.

On ne conclut pas qu'une simple *prior probability shift* a été formellement démontrée ; le résultat est seulement cohérent avec le drift observé.


## 5. Politique économique finale


In [22]:
# Politique économique finale
test_ev = expected_net_value(
    probability=test_probability,
    amount=test["amount"],
    investigation_cost=100.0,
    loss_fraction=1.0,
    intervention_effectiveness=0.8,
)

test_review = (
    test_ev > 0
)

# Les mêmes métriques
test_economic_alerts = int(
    test_review.sum()
)

test_economic_tp = int(
    test.loc[
        test_review,
        "is_fraud"
    ].sum()
)

test_economic_precision = (
    test_economic_tp
    / test_economic_alerts
)

test_economic_recall = (
    test_economic_tp
    / test["is_fraud"].sum()
)

print(
    f"Alertes   : {test_economic_alerts:,}"
)

print(
    f"TP        : {test_economic_tp:,}"
)

print(
    f"Precision : "
    f"{test_economic_precision:.2%}"
)

print(
    f"Recall    : "
    f"{test_economic_recall:.2%}"
)

Alertes   : 6,072
TP        : 3,994
Precision : 65.78%
Recall    : 99.60%


### Conclusion — la politique conserve un rappel élevé

Avec le scénario de référence :
- **6 072 contrôles**
- **3 994 fraudes détectées**
- Precision = **65,78 %**
- Recall = **99,60 %**

La précision est beaucoup plus élevée que sur Validation, principalement parce que la fraude est plus fréquente dans cette période du simulateur. Il ne faut donc pas comparer les précisions des deux périodes sans tenir compte du **base rate**.


## 6. Contrôle d'intégration avant l'application


In [24]:
# Test avant Streamlit
from src.service import (
    FraudScoringService,
)

service = FraudScoringService()

print(
    "Service de scoring chargé : OK"
)

Service de scoring chargé : OK


Le service de scoring se charge correctement avec les artefacts gelés. Aucune donnée de Validation n'est rechargée dans ce notebook final : l'ancienne cellule exploratoire qui référençait `validation` a été retirée car elle n'appartenait pas au protocole du holdout.


### 6.1 Vérification de l'environnement d'exécution


In [1]:
import sys

print(sys.executable)
print(sys.version)

C:\Users\Emmanuel\Documents\Professionnel\Formations\DIT\fraude-mobile-money\.venv312\Scripts\python.exe
3.12.13 (main, Mar  3 2026, 15:01:35) [MSC v.1944 64 bit (AMD64)]


In [2]:
import src
import pandas
import sklearn
import xgboost

print("Kernel projet OK")
print("src :", src.__file__)

Kernel projet OK
src : C:\Users\Emmanuel\Documents\Professionnel\Formations\DIT\fraude-mobile-money\src\__init__.py


# Conclusion finale du holdout

Le Test confirme trois points :

1. **Le ranking H2 généralise très bien sur la période finale de PaySim.**
2. **La règle métier reste extrêmement forte**, mais le résiduel ML apporte de la valeur sur les fraudes atypiques.
3. **La calibration probabiliste dérive davantage que le ranking**, ce qui justifie un monitoring et une recalibration périodique dans un système réel.

Aucun réglage n'est effectué à partir de ces résultats. Les artefacts évalués ici sont ceux qui sont ensuite exposés par `FraudScoringService`, Streamlit et Docker.
